In [ ]:
# Cell 0 — Check all required libraries are installed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")
print(f"NumPy version     : {np.__version__}")
print(f"Pandas version    : {pd.__version__}")
print(f"Seaborn version   : {sns.__version__}")

# If any import fails, install the missing library using:
# pip install scikit-learn numpy pandas matplotlib seaborn
# Then re-run the cell.

In [ ]:
# Cell 1 — Load Dataset
cancer = load_breast_cancer()

# Convert to pandas DataFrame
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Add target column: original 0=malignant, 1=benign
# We FLIP it so 1=malignant, 0=benign (more intuitive medically)
df['target'] = 1 - cancer.target

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Cell 2 — Verify Target Labels
print("Target value counts:")
print(df['target'].value_counts())
print()
print("0 = Benign  (not cancer)")
print("1 = Malignant (cancer)")
print()
print("Class balance:")
total = len(df)
malignant = df['target'].sum()
benign = total - malignant
print(f"Malignant: {malignant} ({malignant/total*100:.1f}%)")
print(f"Benign   : {benign} ({benign/total*100:.1f}%)")

In [ ]:
# Cell 3 — Basic Statistics
print("=== First 5 Rows ===")
print(df.head())
print()
print("=== Dataset Info ===")
print(df.info())
print()
print("=== Statistical Summary ===")
print(df.describe().round(2))
print()
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print(f"Total missing values: {df.isnull().sum().sum()}")

In [ ]:
# Cell 4 — Class Distribution Plot
plt.figure(figsize=(6, 4))

counts = df['target'].value_counts()
colors = ['steelblue', 'tomato']
bars = plt.bar(['Benign (0)', 'Malignant (1)'], 
               [counts[0], counts[1]], 
               color=colors, 
               edgecolor='white',
               width=0.5)

# Add count labels on top of bars
for bar, count in zip(bars, [counts[0], counts[1]]):
    plt.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + 5,
             str(count), 
             ha='center', 
             va='bottom', 
             fontweight='bold',
             fontsize=12)

plt.title('Breast Cancer Dataset — Class Distribution', 
          fontsize=13, fontweight='bold')
plt.ylabel('Number of Samples')
plt.ylim(0, 420)
plt.tight_layout()
plt.savefig('plot1_class_distribution.png', dpi=150)
plt.show()
print("Plot saved as plot1_class_distribution.png")

In [ ]:
# Cell 5 — Correlation Heatmap
plt.figure(figsize=(14, 12))
correlation_matrix = df.corr()

sns.heatmap(
    correlation_matrix,
    annot=False,
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.1,
    cbar_kws={'shrink': 0.8}
)

plt.title('Feature Correlation Matrix — All 30 Features + Target', 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot2_correlation_heatmap.png', dpi=150)
plt.show()
print("Plot saved as plot2_correlation_heatmap.png")

# Print top 10 features most correlated with target
print("\n=== Top 10 Features Most Correlated with Target ===")
target_corr = correlation_matrix['target'].abs().sort_values(
    ascending=False
)
print(target_corr[1:11])

In [ ]:
# Cell 6 — Feature Distributions by Class
# Plot the 4 most important features side by side
top4_features = ['mean radius', 
                 'mean texture',
                 'mean concave points', 
                 'worst radius']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feature in enumerate(top4_features):
    benign_data    = df[df['target'] == 0][feature]
    malignant_data = df[df['target'] == 1][feature]
    
    axes[i].hist(benign_data, bins=25, alpha=0.6, 
                 color='steelblue', label='Benign')
    axes[i].hist(malignant_data, bins=25, alpha=0.6, 
                 color='tomato', label='Malignant')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].legend()

plt.suptitle('Feature Distributions: Benign vs Malignant', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot3_feature_distributions.png', dpi=150)
plt.show()
print("Plot saved as plot3_feature_distributions.png")

## EDA Summary — Key Findings

1. **Dataset Shape:** 569 samples, 30 numeric features, 1 binary target
2. **Missing Values:** None — no imputation required
3. **Class Balance:** 63% Benign, 37% Malignant — mild imbalance, 
   handled by stratified split
4. **Most Predictive Features:** worst concave points, worst perimeter, 
   mean concave points, worst radius show clear separation between classes
5. **Feature Scale Problem:** mean area (~654) vs mean smoothness (~0.096) 
   — StandardScaler is essential before training
6. **Multicollinearity Noted:** radius, perimeter, and area are 
   geometrically related and highly correlated with each other